# Import libraries

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from google import genai
from google.genai import types

from IPython.display import display
from IPython.display import Markdown
import textwrap

from pathlib import Path
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import  GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA

# Initialize the model and client

In [10]:
model = "gemini-3.6-flash"
client = genai.Client()

In [4]:
chat = client.chats.create(
    model=model
)

response = chat.send_message("Explain Generative AI with 3 bullet points")
#print(response.text)
Markdown(response.text)

Here is Generative AI explained in 3 bullet points:

* **Creates New Content:** Unlike traditional AI that analyzes existing data, Generative AI uses advanced machine learning to create entirely new content—including text, images, music, video, and computer code.
* **Learns from Patterns:** It works by analyzing massive datasets to learn underlying patterns, grammar, and structures, allowing it to predict and generate original, human-like outputs when given a text prompt.
* **Transforms Work and Creativity:** It is used across various industries to automate repetitive tasks, assist in creative brainstorming (like design and writing), speed up software development, and solve complex problems.

# Checking on token count

In [7]:
for i in chat.get_history():
    print(i)

Markdown(f"Total token count: {response.usage_metadata.total_token_count}")

parts=[Part(
  text='Explain Generative AI with 3 bullet points'
)] role='user'
parts=[Part(
  text="""Here is Generative AI explained in 3 bullet points:

* **Creates New Content:** Unlike traditional AI that analyzes existing data, Generative AI uses advanced machine learning to create entirely new content—including text, images, music, video, and computer code.
* **Learns from Patterns:** It works by analyzing massive datasets to learn underlying patterns, grammar, and structures, allowing it to predict and generate original, human-like outputs when given a text prompt.
* **Transforms Work and Creativity:** It is used across various industries to automate repetitive tasks, assist in creative brainstorming (like design and writing), speed up software development, and solve complex problems.""",
  thought_signature=b'\x12\x8c\x13\n\x89\x13\x01\x11M2\x0ff\x1cH\xa6\x83\xc5\xe5\xcbYt\xfd=\xc7c\xbc(z*v\xb1\x13\xed\xf6\xef\x84Q\x1c_\xb7\xb4\xe5"\x16\x03\xb7?\x0e\xe6#\xcbi\xf2\xf4P\x97\x14\

Total token count: 678

# Experimenting with Temperature

In [14]:
config = types.GenerateContentConfig(temperature=1.0)

chat = client.chats.create(
    model=model,
    config = config
)

print(client.models.get(model=model).temperature)
response = chat.send_message("Explain Generative AI with 3 bullet points")

Markdown(response.text)



1.0


Here is Generative AI explained in 3 bullet points:

* **Creates New Content:** Unlike traditional AI that analyzes or categorizes existing data, Generative AI produces brand-new, original content—including text, images, music, video, and computer code.
* **Learns from Patterns:** It works by analyzing massive amounts of data to learn underlying patterns and language structures, allowing it to generate relevant, human-like outputs based on simple prompts.
* **Powers Modern Tools:** It is the driving force behind popular applications like ChatGPT, Claude, and Midjourney, transforming how people write, design, code, and solve complex problems.

In [ ]:
config = types.GenerateContentConfig(temperature=1.3)
chat = client.chats.create(
    model=model,
    config = config
)

response = chat.send_message("Explain Generative AI with 3 bullet points")
print(client.models.get(model=model).temperature)

Markdown(response.text)

1.0


Here is an explanation of Generative AI in 3 bullet points:

* **Creates New Content:** Unlike traditional AI that only analyzes or categorizes existing data, Generative AI creates brand-new, original content—including human-like text, images, music, video, and computer code.
* **Learns from Patterns:** It works by analyzing massive amounts of training data to learn underlying patterns and structures, allowing it to predict and generate logical, context-relevant responses based on user prompts.
* **Boosts Productivity and Creativity:** It powers modern tools like conversational chatbots (e.g., ChatGPT), digital art generators, and automated coding assistants, helping humans brainstorm, automate repetitive tasks, and design faster.

##  MaxToken

In [ ]:
# Input token limit
print(client.models.get(model=model).input_token_limit)

# Output token limit
print(client.models.get(model=model).output_token_limit)

# a measure of how many of the most probable tokens are considered at each step. 
# It affects the model’s token selection strategy for generating outputs.
print(client.models.get(model=model).top_k)

# parameter controls how the AI model chooses words when generating text. 
# It looks at words from most to least likely, adding up their probabilities until reaching the top_p value. 
# Then, it picks the next word from this group, with some influence from the temperature parameter. 
# A lower top_p produces more focused and predictable text, while a higher top_p results in less predictable answers.
print(client.models.get(model=model).top_p)

1048576
65536
64
0.95


# Experiment with top_p parameter. Lower top_p produces more focussed and predictable text,

In [3]:
config = types.GenerateContentConfig(top_p=1.0, temperature=1.0)

chat = client.chats.create(
    model=model,
    config = config
)

response = chat.send_message("Explain steps to create a pizza")
print(client.models.get(model=model).temperature)

Markdown(response.text)

1.0


Creating a homemade pizza from scratch is a rewarding process. Here is a step-by-step guide to making a classic, delicious pizza at home, from dough to table.

---

### **Ingredients Overview**

*   **For the Dough:** 3 to 3.5 cups flour (bread flour or all-purpose), 1 packet (2 ¼ tsp) active dry yeast, 1 tsp sugar, 1 ¼ cups warm water, 2 tbsp olive oil, 1 ½ tsp salt.
*   **For the Sauce:** 1 can (15 oz) crushed tomatoes, 1 minced garlic clove, 1 tsp dried oregano, salt, and pepper to taste.
*   **For the Toppings:** 2 cups shredded low-moisture mozzarella cheese, plus your choice of meats (pepperoni, sausage) and vegetables (bell peppers, mushrooms, onions, basil).

---

### **Step 1: Make the Dough**
1.  **Activate the Yeast:** In a bowl, combine the warm water (about 105°F–110°F), sugar, and yeast. Let it sit for 5–10 minutes until it becomes foamy.
2.  **Mix the Ingredients:** Add the olive oil, salt, and 2 cups of flour to the yeast mixture. Stir until combined. Gradually add the remaining flour until a sticky dough forms.
3.  **Knead the Dough:** Turn the dough onto a floured surface. Knead for 8–10 minutes until it becomes smooth and elastic. (If it’s too sticky, sprinkle a little extra flour).
4.  **Let it Rise:** Lightly oil a large bowl, place the dough inside, and cover it with a damp towel or plastic wrap. Place it in a warm spot for **1 to 1.5 hours**, or until it doubles in size.

---

### **Step 2: Prepare the Sauce & Toppings**
1.  **Make the Sauce:** In a small bowl, mix the crushed tomatoes, garlic, oregano, salt, and pepper. (No need to cook the sauce beforehand; it will cook in the oven).
2.  **Prep Toppings:** Chop your vegetables and meats. 
    *   *Pro-Tip:* Pre-cook watery vegetables (like mushrooms or onions) slightly in a pan to prevent your pizza crust from getting soggy.

---

### **Step 3: Preheat the Oven**
1.  **Set the Heat High:** Preheat your oven to its highest setting—usually between **475°F to 500°F (245°C to 260°C)**.
2.  **Preheat the Surface:** If using a pizza stone or baking steel, place it in the oven while it preheats. If using a standard baking sheet, grease it lightly or line it with parchment paper.

---

### **Step 4: Shape the Pizza**
1.  **Punch Down Dough:** Gently press down on the risen dough to release air bubbles. Divide it into two balls if you want thinner pizzas, or keep it as one for a larger, thicker crust.
2.  **Stretch the Dough:** On a surface dusted with flour or cornmeal, gently press and stretch the dough out into a circle (about 12 inches) using your hands. *Avoid using a rolling pin, as it flattens the air pockets needed for a light crust.*
3.  **Transfer:** Place the stretched dough onto a parchment sheet or a cornmeal-dusted pizza peel/baking tray.

---

### **Step 5: Assemble the Pizza**
1.  **Add Sauce:** Ladle a few spoonfuls of sauce into the center of the dough and spread it outward in a circular motion, leaving a 1/2-inch border for the crust.
2.  **Add Cheese:** Evenly sprinkle the shredded mozzarella over the sauce.
3.  **Add Toppings:** Layer your desired toppings on top. 
    *   *Rule of thumb:* Less is more! Overloading the pizza with toppings will result in a soggy, undercooked crust.

---

### **Step 6: Bake the Pizza**
1.  **Bake:** Transfer the pizza to the oven. Bake for **10 to 15 minutes** (or 8–10 minutes if using a pizza stone). 
2.  **Check Doneness:** The pizza is ready when the crust is golden-brown and the cheese is melted and bubbling with small brown spots.

---

### **Step 7: Slice and Serve**
1.  **Cool:** Remove the pizza from the oven and let it sit on a cutting board for **2–3 minutes**. This allows the cheese to set so it doesn’t slide off when cut.
2.  **Garnish:** Add fresh herbs (like fresh basil leaves), grated Parmesan cheese, or red pepper flakes if desired.
3.  **Slice:** Use a pizza cutter or a large chef's knife to slice into triangles and enjoy!

In [4]:
config = types.GenerateContentConfig(top_p=0, temperature=1.0)

chat = client.chats.create(
    model=model,
    config = config
)

response = chat.send_message("Explain steps to create a pizza")
print(client.models.get(model=model).temperature)

Markdown(response.text)

1.0


Making a delicious pizza at home from scratch is a rewarding process. Here is a step-by-step guide to making a classic, homemade cheese or pepperoni pizza.

---

### **Ingredients Overview**

*   **For the Dough:** 2 to 2.5 cups flour (bread flour or all-purpose), 1 packet (2 1/4 tsp) active dry yeast, 1 tsp sugar, 1 tsp salt, 3/4 cup warm water, 1 tbsp olive oil.
*   **For the Sauce:** 1 cup crushed canned tomatoes, 1 clove garlic (minced), 1 tsp dried oregano, salt, and pepper to taste.
*   **For the Toppings:** 1.5 cups shredded low-moisture mozzarella cheese, plus your choice of toppings (pepperoni, mushrooms, fresh basil, etc.).

---

### **Step 1: Make the Dough**

1.  **Activate the Yeast:** In a bowl, mix warm water (about 110°F/43°C), sugar, and yeast. Let it sit for 5–10 minutes until it becomes foamy.
2.  **Mix the Ingredients:** Add the olive oil, salt, and half of the flour to the yeast mixture. Stir well, then gradually add the rest of the flour until a manageable dough forms.
3.  **Knead:** Transfer the dough to a floured surface. Knead for 8–10 minutes until smooth and elastic. (If it sticks to your hands, add a sprinkle of flour).
4.  **First Rise:** Coat a clean bowl with a little olive oil. Place the dough inside, cover it with a damp towel or plastic wrap, and let it rise in a warm spot for **1 to 1.5 hours** (or until doubled in size).

---

### **Step 2: Prepare the Oven & Toppings**

1.  **Preheat the Oven:** Preheat your oven to its highest setting—usually **475°F to 500°F (245°C to 260°C)**. If using a pizza stone or steel, place it on the middle rack now. 
2.  **Make the Sauce:** In a small bowl, combine the crushed tomatoes, minced garlic, oregano, salt, and pepper. (No cooking needed; it will cook on the pizza).
3.  **Prep Toppings:** Shred your cheese and slice any vegetables or meats. 
    *   *Tip:* Pre-cook watery vegetables like mushrooms or spinach so they don't make your crust soggy.

---

### **Step 3: Shape the Pizza**

1.  **Punch Down Dough:** Gently press the risen dough to release air bubbles.
2.  **Stretch the Dough:** On a piece of parchment paper or a surface dusted with cornmeal/flour, gently press and stretch the dough outward into a 12-inch circle. 
    *   *Tip:* Leave the edges slightly thicker to create a crust rim. Try to stretch by hand rather than rolling with a pin to keep it airy.

---

### **Step 4: Assemble the Pizza**

1.  **Add Sauce:** Spoon a light layer of sauce onto the dough, leaving a 1/2-inch border around the edge for the crust. *(Don't use too much sauce, or the crust will get soggy).*
2.  **Add Cheese:** Evenly sprinkle the shredded mozzarella over the sauce.
3.  **Add Toppings:** Place your toppings (like pepperoni or peppers) on top of the cheese. *(Less is more—too many toppings prevent the dough from cooking through).*

---

### **Step 5: Bake and Serve**

1.  **Bake:** Transfer the pizza (with the parchment paper underneath, if using) onto your preheated baking sheet or pizza stone. Bake for **10 to 15 minutes**, or until the crust is golden brown and the cheese is melted and bubbling.
2.  **Rest:** Remove from the oven and let it sit for **2 to 3 minutes**. This allows the cheese to set so it doesn't slide off when sliced.
3.  **Garnish and Slice:** Add fresh herbs (like fresh basil) or a sprinkle of crushed red pepper flakes, slice, and enjoy!

---

### **Pro-Tips for Success:**
*   **High Heat is Key:** Home ovens don't get as hot as commercial pizza ovens, so maximum heat helps create a crispy crust.
*   **Use Low-Moisture Cheese:** Fresh mozzarella holds a lot of water and can make the pizza wet. Pre-shredded, low-moisture mozzarella works best for classic home pizzas.
*   **Cornmeal for Crunch:** Dusting your baking tray with coarse cornmeal gives the bottom crust a great crunch and prevents sticking.

# Experiment with the candidate_count parameter

In [5]:
config = types.GenerateContentConfig(candidate_count=1)

chat = client.chats.create(model=model,
                           config=config
)

response = chat.send_message("Explain steps to create a pizza")

In [6]:
Markdown(response.text)

Making a pizza from scratch is fun, rewarding, and yields a far better result than frozen options. Here is a comprehensive, step-by-step guide to making a classic homemade pizza.

---

### **Ingredients Checklist**

#### **For the Dough:**
*   **Warm Water:** 1 cup (240ml) – around 105°F to 110°F (warm, not hot)
*   **Active Dry Yeast:** 1 packet (2 ¼ teaspoons)
*   **Sugar or Honey:** 1 teaspoon (helps activate yeast)
*   **Bread Flour or All-Purpose Flour:** 2 ½ to 3 cups
*   **Olive Oil:** 2 tablespoons (plus extra for oiling)
*   **Salt:** 1 teaspoon

#### **For the Pizza:**
*   **Pizza Sauce:** ½ to ¾ cup (store-bought or homemade tomato sauce)
*   **Cheese:** 1 ½ to 2 cups shredded low-moisture mozzarella (shredding your own block melts best)
*   **Toppings of choice:** Pepperoni, sliced mushrooms, bell peppers, onions, fresh basil, cooked sausage, etc.

---

### **Step 1: Make the Dough**

1.  **Activate the Yeast:** 
    In a large bowl, combine warm water, sugar, and yeast. Stir gently and let it sit for 5–10 minutes until it becomes foamy on top.
2.  **Mix the Dough:** 
    Add the olive oil, salt, and 2 cups of flour to the yeast mixture. Stir with a wooden spoon until combined. Gradually add the remaining flour until the dough pulls away from the sides of the bowl and isn't overly sticky.
3.  **Knead:** 
    Turn the dough onto a floured surface. Knead for 8–10 minutes by pushing the dough away with the heel of your hand, folding it back over, and repeating. The dough should become smooth, elastic, and spring back when poked.
4.  **Let it Rise:** 
    Lightly oil a clean bowl. Place the dough ball inside, turning it once to coat it in oil. Cover the bowl with a damp towel or plastic wrap. Place it in a warm, draft-free spot for **1 to 1.5 hours**, or until it doubles in size.

---

### **Step 2: Prep Your Oven and Ingredients**

1.  **Preheat High:** 
    Preheat your oven to its highest setting—usually **475°F to 500°F (245°C to 260°C)**. *High heat is secret to a good pizza crust.*
    *   *Note:* If using a pizza stone or steel, place it on the middle rack before preheating.
2.  **Prepare Toppings:** 
    Shred your cheese and chop your vegetables. Cook any raw meats (like Italian sausage or bacon) ahead of time. 

---

### **Step 3: Shape the Dough**

1.  **Punch Down:** 
    Once risen, gently punch the dough down to release air bubbles. (This recipe makes one thick 12-inch pizza, or two thin 10-inch pizzas).
2.  **Stretch:** 
    On a sheet of parchment paper or a lightly floured surface, press the dough out with your fingers starting from the center and working outward. Leave a thicker ring around the edges for the crust. 
    *   *Tip:* Avoid using a rolling pin if you want an airy crust, as it squeezes out the air pockets.

---

### **Step 4: Assemble the Pizza**

1.  **Dock the Dough:** Use a fork to prick the center area of the dough lightly (this prevents large air bubbles from swelling during baking).
2.  **Add Sauce:** Spoon the sauce into the center and spread it outward using the back of the spoon, leaving a ½-inch border around the edge.
3.  **Add Cheese:** Evenly sprinkle the mozzarella over the sauce.
4.  **Add Toppings:** Layer your toppings over the cheese. 
    *   *Rule of thumb:* Don't overload the pizza, or the crust will become soggy.

---

### **Step 5: Bake**

1.  **Transfer:** If using a pizza stone, use a pizza peel (or the back of a baking sheet) to slide the parchment paper with the pizza onto the stone. Otherwise, place your pizza directly on a rimmed baking sheet.
2.  **Bake:** Bake at 475°F–500°F for **10 to 15 minutes**. 
3.  **Check Doneness:** The pizza is ready when the crust is golden brown and the cheese is bubbly and melted with slight browning spots.

---

### **Step 6: Finish and Serve**

1.  **Cool:** Remove the pizza from the oven and let it sit on a cutting board for **2 to 3 minutes**. (If you cut it immediately, the hot cheese will slide right off).
2.  **Garnish:** Add fresh herbs like basil, a sprinkle of parmesan, or red pepper flakes.
3.  **Slice:** Cut into 8 triangles using a pizza cutter or a large chef's knife. Enjoy!

---

### **Pro-Tips for Success:**
*   **Block Cheese over Bagged:** Pre-shredded cheese in bags contains anti-caking agents that prevent it from melting smoothly. Shred your own for best results.
*   **Cold Fermentation (Optional):** For deep, bakery-quality flavor, make the dough a day ahead and let it rise slowly in the refrigerator for 24–48 hours instead of 1 hour at room temperature. Bring it back to room temp before shaping.

# Introduction to RAG

# Load the PDF and extract the texts

In [2]:
# 1. Load a PDF to answer questions
# 2. Create a text splitter using RecursiveCharacterTextSpltter

file_name = "Sample_doc.pdf"
reader = PdfReader(file_name)

pages = len(reader.pages)

print(pages)
text_with_page = {}

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    length_function = len
)

for index, page in enumerate(reader.pages):
    text_with_page[index] = page.extract_text()
#print(text_with_page)

chunk_with_page_num = {}

for page_num, text in text_with_page.items():
    chunks = text_splitter.split_text(text=text)
    chunk_with_page_num[page_num] = chunks

for page_num, chunks in chunk_with_page_num.items():
    print(page_num, chunks)



Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 67 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)


3
0 ['9. Details of performance and career development reviews of employees and worker:\nCategory\nFY 2024-25 FY 2023-24\nTotal (A) No. (B) % (B/A) Total (C) No. (D) % (D/C)\nEmployees\nMale 36,796 36,796 100 36,198 36,198 100\nFemale 2,362 2,362 100 2,046 2,046 100\nTotal 39,158 39,158 100 38,244 38,244 100\nWorkers\nMale 3,493 3,493 100 3,855 3,855 100\nFemale 52 52 100 67 67 100\nTotal 3,545 3,545 100 3,922 3,922 100\nESSENTIAL INDICA TORS\nSECTION C: PRINCIPLE-WISE PERFORMANCE DISCLOSURE\nPRINCIPLE 3\n6.  Is there a mechanism available to receive and redress grievances for the following categories of \nemployees and workers? If yes, give details of the mechanism in brief.\nYes/No (If yes, then give details of the mechanism in brief)\nPermanent Workers Yes. Grievance procedures are defined for each location with unionised workforce. They are also privy to the \navailable multiple channels of grievance redressal. The Company has a Vigil Mechanism and Whistle-blower', 'policy under wh

# Create embeddings using GoogleGenerativeAIEmbeddings

In [3]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [4]:
for page_num, chunks in chunk_with_page_num.items():
    embeddings.embed_documents(chunks)

In [7]:
# Create a vector database and a retriever
vector_index = Chroma.from_texts(chunks, embeddings)

retriever = vector_index.as_retriever(search_kwargs={"k" : 5})

# Create a RAG Chain and Ask query

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
#from langchain.chain

model = "gemini-3.6-flash"
#client = genai.Client()

llm = ChatGoogleGenerativeAI(
    model=model
)

# Generate the RAG question answer generating chain

qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = retriever,
    return_source_documents=True
)

# Ask a question and get an answer
question = "Summarize the details or safety related incidents"

result = qa_chain.invoke({"query": question})


In [22]:
Markdown(result["result"])

Based on the provided context, here is a summary of the details regarding safety-related incidents and practices:

### 1. Work-Related Incidents & Statistics
* **Affected Workers (FY 2024-25):** 3 workers suffered high-consequence work-related injuries, ill health, or fatalities (0 employees). None were rehabilitated or placed in suitable employment during the period.
* **Affected Workers (FY 2023-24):** 1 worker suffered a high-consequence work-related injury, ill health, or fatality (0 employees). None were rehabilitated or placed.
* **Incident Causes & Preventative Action:** 
  * Fatality counts include traffic incidents involving material handling equipment.
  * To prevent the recurrence of similar incidents, reverse cameras have been installed on all major material handling equipment.

### 2. Marine & Ship Chartering Safety Vetting
* **Vetting Process:** Ships taken on spot or time charter undergo third-party and in-house marine vetting based on past incident details, inspection reports, operator records, and crew experience.
* **Risk Mitigation:** If safety concerns are identified during vetting, counterparties are required to submit a detailed risk mitigation analysis before any contract is finalized.

### 3. Supply Chain / Value Chain Safety
* EcoVadis assessments identified health, safety practice, and working condition concerns among value chain partners.